In [1]:
# ===== Cell 1: Imports =====
import os
import time
import torch
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from pathlib import Path
import xlsxwriter
from datetime import timedelta
import requests

# ===== End of Cell 1 =====

C:\Users\omrym\anaconda3\envs\deep_learn\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# ===== Cell 2: Automated Online Data Acquisition (Tiingo) =====

# --- 1. Configuration ---
TIINGO_API_KEY = "5cbffd02e11af8771393bbeb57a4b3a90de96f8e"
SAVE_DIR = Path("../data/raw_data")
os.makedirs(SAVE_DIR, exist_ok=True)

# Fetching from 2019 to ensure enough lookback for a 2020 training start
FETCH_START = "2019-01-01"
FETCH_END = "2025-12-31"

# Matching your Production List (48 Stocks)
stocks_to_fetch = [
    "AAPL", "ABBV", "ADBE", "AMD", "AMT", "AMZN", "AVGO", "BAC", "BHP", "BLK",
    "BP", "COP", "COST", "CVX", "DLR", "EQIX", "FCX", "GOOGL", "GS", "INTC",
    "JNJ", "JPM", "LIN", "LLY", "META", "MRK", "MS", "NEM", "NFLX", "NVDA",
    "O", "PEP", "PFE", "PLD", "QCOM", "RIO", "SCCO", "SCHW", "SHW", "SLB",
    "SPG", "TMO", "TSLA", "TTE", "UNH", "WELL", "WFC", "XOM"
]

def fetch_and_save_raw(ticker):
    file_path = SAVE_DIR / f"{ticker}_RAW.csv"

    # Skip if already exists to save API credits and time
    if file_path.exists():
        return True

    headers = {'Content-Type': 'application/json'}
    url = f"https://api.tiingo.com/tiingo/daily/{ticker}/prices"
    params = {
        'startDate': FETCH_START,
        'endDate': FETCH_END,
        'token': TIINGO_API_KEY,
        'format': 'json'
    }

    try:
        response = requests.get(url, params=params, headers=headers)
        if response.status_code == 200:
            raw_data = response.json()
            if not raw_data: return False

            df = pd.DataFrame(raw_data)
            # Select Adjusted columns for splits/dividends
            clean_df = df[['date', 'adjOpen', 'adjHigh', 'adjLow', 'adjClose', 'adjVolume']].copy()
            clean_df.columns = ['Date', 'Open', 'High', 'Low', 'Close', 'Volume']
            clean_df['Date'] = pd.to_datetime(clean_df['Date']).dt.date

            clean_df.to_csv(file_path, index=False)
            print(f"✅ {ticker}: {len(clean_df)} days saved.")
            return True
        else:
            print(f"❌ Error {response.status_code} for {ticker}")
            return False
    except Exception as e:
        print(f"⚠️ Exception fetching {ticker}: {e}")
        return False

# --- 2. Execution ---
print(f"🚀 Verifying/Fetching raw data for {len(stocks_to_fetch)} stocks...")
for i, ticker in enumerate(stocks_to_fetch):
    success = fetch_and_save_raw(ticker)
    # Tiingo Rate Limit Protection
    if i < len(stocks_to_fetch) - 1 and success:
        time.sleep(3) # Small sleep for verification; Tiingo allows 50/hour or 2/sec depending on tier

print(f"🏁 Data Directory Synchronized.")

# ===== End of Cell 2 =====

In [2]:
# ===== Cell 3: Final Production - Raw/Processed & Standardized M=55 Data =====

# --- 1. CONFIGURATION ---
M_TRADING_DAYS = 55
TARGET_DAYS = 5
ATR_MULTIPLIER = 1.5
EMBEDDING_DIM = 8
START_DATE = "2020-01-01"
END_DATE = "2025-12-31"

STOCKS_LIST = [
    "AAPL", "ABBV", "ADBE", "AMD", "AMT", "AMZN", "AVGO", "BAC", "BHP", "BLK",
    "BP", "COP", "COST", "CVX", "DLR", "EQIX", "FCX", "GOOGL", "GS", "INTC",
    "JNJ", "JPM", "LIN", "LLY", "META", "MRK", "MS", "NEM", "NFLX", "NVDA",
    "O", "PEP", "PFE", "PLD", "QCOM", "RIO", "SCCO", "SCHW", "SHW", "SLB",
    "SPG", "TMO", "TSLA", "TTE", "UNH", "WELL", "WFC", "XOM"
]

BASE_DATA = Path("../data")
INPUT_DIR = BASE_DATA / "raw_data"
PROCESSED_EXCEL_DIR = BASE_DATA / "formatted_data" / "atr_1.5_processed"
RAW_EXCEL_DIR = BASE_DATA / "formatted_data" / "atr_1.5_raw"
MASTER_FILE_PATH = BASE_DATA / "processed_data" / "master_dataset_M55_trading.pt"

for d in [PROCESSED_EXCEL_DIR, RAW_EXCEL_DIR, MASTER_FILE_PATH.parent]:
    os.makedirs(d, exist_ok=True)

# Normalization Columns
Z_COLS = ['O_pct', 'H_pct', 'L_pct', 'C_pct', 'MA_Mom']
RANGE_0_1 = ['V_money', 'ATR_pct']
FINAL_FEATURES = ['O_pct', 'H_pct', 'L_pct', 'C_pct', 'V_norm', 'MA_Mom', 'ATR_pct_norm']

headers = ['Num', 'Date', 'Gap(O)', 'UpperW(H)', 'LowerW(L)', 'Body(C)', 'V_Money_Norm', 'MA_Mom', 'ATR_pct', ''] + \
          [f"Emb_{i+1}" for i in range(EMBEDDING_DIM)] + ['', 'Label', 'Start Target Date', 'End Target Date']

# --- 2. HELPERS ---

def get_inflation_factor(year):
    factors = {2017: 1.35, 2018: 1.32, 2019: 1.29, 2020: 1.27, 2021: 1.21,
               2022: 1.12, 2023: 1.08, 2024: 1.03, 2025: 1.00}
    return factors.get(year, 1.0)

def compute_rigorous_features(df):
    ma_period = 14
    df['MA_Abs'] = df['Close'].rolling(window=ma_period).mean()
    tr = pd.concat([(df['High']-df['Low']), (df['High']-df['Close'].shift()).abs(), (df['Low']-df['Close'].shift()).abs()], axis=1).max(axis=1)
    df['ATR_Abs'] = tr.rolling(window=ma_period).mean()
    # Rigorous Feature Logic
    df['O_pct'] = (df['Open'] / df['Close'].shift(1)) - 1
    df['H_pct'] = (df['High'] / df['Open']) - 1
    df['L_pct'] = (df['Low'] / df['Open']) - 1
    df['C_pct'] = (df['Close'] / df['Open']) - 1
    df['MA_Mom'] = (df['MA_Abs'] / df['MA_Abs'].shift(1)) - 1
    df['ATR_pct'] = df['ATR_Abs'] / df['Close']
    df['Year'] = pd.to_datetime(df['Date']).dt.year
    df['V_money'] = (df['Volume'] * df['Close']) * df['Year'].apply(get_inflation_factor)
    return df

def get_window_label_rigorous(df, t, multiplier):
    atr_t = df.iloc[t]['ATR_Abs']
    open_next = df.iloc[t+1]['Open']
    buy_lvl, sell_lvl = open_next + (atr_t * multiplier), open_next - (atr_t * multiplier)
    label, is_v, trigger_idx, trigger_type = 1, False, None, None
    for k in range(1, 6):
        idx = t + k
        h, l, c = df.iloc[idx][['High', 'Low', 'Close']]
        hb, ls = h >= buy_lvl, l <= sell_lvl
        if hb and ls:
            is_v = True
            trigger_idx, trigger_type, label = idx, 'both', (0 if abs(h - c) < abs(l - c) else 2)
            break
        elif hb:
            trigger_idx, trigger_type, label = idx, 'buy', 2
            break
        elif ls:
            trigger_idx, trigger_type, label = idx, 'sell', 0
            break
    return label, is_v, trigger_idx, trigger_type

# --- 3. PASS 1: GLOBAL STATS (TRIPLE CHECKED) ---
print("📊 Pass 1: Calculating Global Min/Max for Scaling Consistency...")
all_dfs = []
for ticker_file in INPUT_DIR.glob("*_RAW.csv"):
    df = compute_rigorous_features(pd.read_csv(ticker_file)).dropna()
    mask = (pd.to_datetime(df['Date']) >= pd.Timestamp(START_DATE)) & (pd.to_datetime(df['Date']) <= pd.Timestamp(END_DATE))
    all_dfs.append(df.loc[mask])
global_all = pd.concat(all_dfs)
g_min = global_all[Z_COLS + RANGE_0_1].min()
g_max = global_all[Z_COLS + RANGE_0_1].max()

# --- 4. PASS 2: AUDIT & MASTER COMPILATION ---
print(f"🚀 Pass 2: Processing {len(STOCKS_LIST)} Stocks (M={M_TRADING_DAYS})...")
master_samples = []

for ticker in tqdm(STOCKS_LIST, colour='green'):
    path = INPUT_DIR / f"{ticker}_RAW.csv"
    if not path.exists(): continue

    df_raw_csv = pd.read_csv(path).dropna()
    df_raw_csv['Date'] = pd.to_datetime(df_raw_csv['Date'])
    df_logic = compute_rigorous_features(df_raw_csv).dropna().reset_index(drop=True)

    # Master Normalization (Triple checked logic for .pt file)
    df_proc_master = df_logic.copy()
    for col in Z_COLS: # Scaled [-1, 1]
        df_proc_master[col] = 2 * (df_logic[col] - g_min[col]) / (g_max[col] - g_min[col]) - 1
    df_proc_master['V_norm'] = (df_logic['V_money'] - g_min['V_money']) / (g_max['V_money'] - g_min['V_money'])
    df_proc_master['ATR_pct_norm'] = (df_logic['ATR_pct'] - g_min['ATR_pct']) / (g_max['ATR_pct'] - g_min['ATR_pct'])

    for mode in ['raw', 'processed']:
        filename = f"{mode}_{ATR_MULTIPLIER}_{ticker}.xlsx"
        dir_path = RAW_EXCEL_DIR if mode == 'raw' else PROCESSED_EXCEL_DIR
        writer = pd.ExcelWriter(dir_path / filename, engine='xlsxwriter')
        wb, ws = writer.book, writer.book.add_worksheet('Verification')

        fmts = {
            'head': wb.add_format({'bold': True, 'bg_color': '#D7E4BC', 'border': 1, 'align': 'center'}),
            'data': wb.add_format({'align': 'center', 'valign': 'vcenter'}),
            'special': wb.add_format({'bold': True, 'font_size': 14, 'align': 'center', 'valign': 'vcenter', 'border': 1}),
            'special_blue': wb.add_format({'bold': True, 'font_size': 14, 'align': 'center', 'valign': 'vcenter', 'border': 1, 'bg_color': '#DEEBF7'}),
            'ref_open': wb.add_format({'bold': True, 'font_size': 14, 'bg_color': '#FFFFCC', 'align': 'center', 'valign': 'vcenter', 'border': 1}),
            'buy_audit': wb.add_format({'bold': True, 'bg_color': '#C6EFCE', 'font_color': '#006100', 'align': 'center', 'valign': 'vcenter', 'border': 1, 'text_wrap': True}),
            'sell_audit': wb.add_format({'bold': True, 'bg_color': '#FFC7CE', 'font_color': '#9C0006', 'align': 'center', 'valign': 'vcenter', 'border': 1, 'text_wrap': True}),
            'target_base': wb.add_format({'bg_color': '#F2F2F2', 'align': 'center', 'valign': 'vcenter'}),
            'buy_win': wb.add_format({'bg_color': '#C6EFCE', 'font_color': '#006100', 'align': 'center', 'bold': True, 'border': 1}),
            'sell_win': wb.add_format({'bg_color': '#FFC7CE', 'font_color': '#9C0006', 'align': 'center', 'bold': True, 'border': 1}),
            'loser_blue': wb.add_format({'bg_color': '#DEEBF7', 'font_color': '#0070C0', 'align': 'center', 'valign': 'vcenter', 'border': 1})
        }

        # Compact Widths
        ws.set_column('A:A', 12); ws.set_column('B:B', 15); ws.set_column('C:I', 15); ws.set_column('T:V', 32)
        for col, val in enumerate(headers): ws.write(0, col, val, fmts['head'])

        curr_row, win_count = 1, 1
        for t in range(M_TRADING_DAYS - 1, len(df_logic) - TARGET_DAYS):
            if df_logic.iloc[t]['Date'] < pd.Timestamp(START_DATE) or df_logic.iloc[t]['Date'] > pd.Timestamp(END_DATE): continue

            label, is_v, trig_idx, trig_type = get_window_label_rigorous(df_logic, t, ATR_MULTIPLIER)
            start_merge = curr_row

            # Write M=55 Windows
            for i in range(t - (M_TRADING_DAYS - 1), t + 1):
                d_logic_row = df_logic.iloc[i]

                # RAW MODE: Physical Values from CSV | PROCESSED MODE: Rigorous % Values
                if mode == 'raw':
                    # Use actual ATR_Abs for raw files
                    feat_vals = [d_logic_row['Open'], d_logic_row['High'], d_logic_row['Low'], d_logic_row['Close'], d_logic_row['Volume'], d_logic_row['MA_Mom'], d_logic_row['ATR_Abs']]
                else:
                    # Use ATR_pct for processed files
                    feat_vals = [d_logic_row['O_pct'], d_logic_row['H_pct'], d_logic_row['L_pct'], d_logic_row['C_pct'], d_logic_row['V_money'], d_logic_row['MA_Mom'], d_logic_row['ATR_pct']]

                ws.write(curr_row, 1, d_logic_row['Date'].strftime('%Y-%m-%d'), fmts['data'])
                ws.write_row(curr_row, 2, feat_vals, fmts['data'])
                ws.write_row(curr_row, 10, [0]*EMBEDDING_DIM, fmts['data'])
                curr_row += 1

            ws.merge_range(start_merge, 0, curr_row-1, 0, f"{win_count} {'V' if is_v else ''}", fmts['special_blue'] if is_v else fmts['special'])
            ws.merge_range(start_merge, 19, curr_row-1, 19, label, fmts['special'])
            ws.merge_range(start_merge, 20, curr_row-1, 20, df_logic.iloc[t+1]['Date'].strftime('%Y-%m-%d'), fmts['special'])
            ws.merge_range(start_merge, 21, curr_row-1, 21, df_logic.iloc[t+5]['Date'].strftime('%Y-%m-%d'), fmts['special'])

            # 5 Target Days (Rigorous audit)
            target_start = curr_row
            t_open_ref, t_atr_ref = df_logic.iloc[t+1]['Open'], df_logic.iloc[t]['ATR_Abs']
            buy_p, sell_p = t_open_ref + (t_atr_ref * ATR_MULTIPLIER), t_open_ref - (t_atr_ref * ATR_MULTIPLIER)

            for k in range(1, TARGET_DAYS + 1):
                idx = t + k
                d_logic_row = df_logic.iloc[idx]

                if mode == 'raw':
                    # Use actual ATR_Abs for raw files
                    feat_vals = [d_logic_row['Open'], d_logic_row['High'], d_logic_row['Low'], d_logic_row['Close'], d_logic_row['Volume'], d_logic_row['MA_Mom'], d_logic_row['ATR_Abs']]
                else:
                    # Use ATR_pct for processed files
                    feat_vals = [d_logic_row['O_pct'], d_logic_row['H_pct'], d_logic_row['L_pct'], d_logic_row['C_pct'], d_logic_row['V_money'], d_logic_row['MA_Mom'], d_logic_row['ATR_pct']]

                ws.write(curr_row, 0, "TARGET", fmts['data'])
                ws.write(curr_row, 1, d_logic_row['Date'].strftime('%Y-%m-%d'), fmts['target_base'])
                for j, val in enumerate(feat_vals):
                    fmt = fmts['target_base']
                    if k == 1 and j == 0: fmt = fmts['ref_open']
                    if idx == trig_idx:
                        if (trig_type == 'buy' and j == 1) or (trig_type == 'both' and label == 2 and j == 1): fmt = fmts['buy_win']
                        elif (trig_type == 'sell' and j == 2) or (trig_type == 'both' and label == 0 and j == 2): fmt = fmts['sell_win']
                        elif trig_type == 'both' and ((label == 2 and j == 2) or (label == 0 and j == 1)): fmt = fmts['loser_blue']
                    ws.write(curr_row, 2+j, val, fmt)
                ws.write_row(curr_row, 10, [0]*EMBEDDING_DIM, fmts['target_base'])
                curr_row += 1

            ws.merge_range(target_start, 19, target_start+1, 21, f"B > {buy_p:.2f}", fmts['buy_audit'])
            ws.merge_range(target_start+3, 19, target_start+4, 21, f"S < {sell_p:.2f}", fmts['sell_audit'])
            ws.set_row(target_start, None, None, {'level': 0, 'collapse': False}) # Force centered display if needed

            if mode == 'processed':
                # Compile final scaled master sample
                master_samples.append({'x': df_proc_master.iloc[t-(M_TRADING_DAYS-1):t+1][FINAL_FEATURES].values.astype(np.float32),
                                       'y': label, 'ticker': ticker, 'date': df_logic.iloc[t]['Date']})
            curr_row += 1; win_count += 1
        writer.close()

torch.save(master_samples, MASTER_FILE_PATH)
print(f"✅ FINALIZED: Raw (Actual ATR) and Processed (ATR %) audits matched. Master saved with {len(master_samples)} windows.")

# ===== End of Cell =====

📊 Pass 1: Calculating Global Min/Max for Scaling Consistency...
🚀 Pass 2: Processing 12 Stocks (M=55)...


100%|██████████| 12/12 [10:17<00:00, 51.48s/it]


✅ FINALIZED: Raw (Actual ATR) and Processed (ATR %) audits matched. Master saved with 18035 windows.
